# 1. Setup
**Approach 2: Feature disentanglement.** The backbone feature is split into a crop part and a disease part.
Adversarial heads (with gradient reversal) try to read the disease from the crop part, and the crop
from the disease part. The encoders learn to defeat them, so each part keeps only its own concept.
An orthogonality loss also stops the two parts from carrying the same information.
Idea: if disease features hold no crop information, a seen disease is easier to spot on a new crop.

In [ ]:
import gc
import sys
from pathlib import Path

import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display

# Find the project root
ROOT = Path.cwd()
if not (ROOT / "common").exists():
    ROOT = ROOT.parent
if not (ROOT / "common").exists():
    raise RuntimeError("Run this notebook from the project root or approaches folder")
sys.path.insert(0, str(ROOT))

from common import config
from common.config import (
    CHECKPOINT_DIR,
    DEVICE,
    NUM_CROPS,
    NUM_DISEASES,
    RESULTS_DIR,
    set_seed,
)
from common.dataset import get_loaders, get_pairs
from common.evaluate import evaluate, val_accuracy

# Use True for a quick local test
DEBUG = True

# Use batch 16 locally and batch 64 on the school RTX 5090
HP = {
    "backbone": "resnet50",
    "lr": 1e-4,
    "weight_decay": 1e-4,
    "epochs": 20,
    "batch_size": 16 if DEBUG else 64,
    "concept_dim": 512,
    "label_smoothing": 0.1,
    "optimizer": "AdamW",
}

# Experiment name orthogonal weight adversarial weight
EXPERIMENTS = [
    ("exp1_encoders", 0.0, 0.0),
    ("exp2_orthogonal", 0.1, 0.0),
    ("exp3_adversarial", 0.0, 0.3),
    ("exp4_combined", 0.1, 0.3),
]

EPOCHS = 1 if DEBUG else HP["epochs"]
RUN_TYPE = "debug" if DEBUG else "full"
USE_AMP = DEVICE.type == "cuda"

print("Device:", DEVICE)
print("Run type:", RUN_TYPE)
print("Epochs:", EPOCHS)
print("Batch size:", HP["batch_size"])
print("Mixed precision:", USE_AMP)

# 2. Data

In [ ]:
# Check the dataset paths before starting any training
required_paths = [
    config.IMAGE_DIR,
    config.TRAIN_CSV,
    config.TEST_SEEN_CSV,
    config.TEST_UNSEEN_CSV,
]
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    missing_text = "\n".join(str(path) for path in missing_paths)
    raise FileNotFoundError(f"Missing dataset paths\n{missing_text}")

pairs = get_pairs().to(DEVICE)


def make_loaders(debug):
    # Create fresh loaders so every experiment starts with the same shuffle state
    return get_loaders(
        HP["batch_size"],
        norm="imagenet",
        debug=debug,
    )


print("Valid crop disease pairs:", len(pairs))
print("Image folder:", config.IMAGE_DIR)

# 3. Model

In [ ]:
class GradientReversal(torch.autograd.Function):
    # Keep features unchanged during the forward pass
    @staticmethod
    def forward(ctx, x):
        return x.view_as(x)

    # Reverse gradients during the backward pass
    @staticmethod
    def backward(ctx, grad):
        return -grad


class DisentangleNet(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()

        # Extract shared image features
        self.backbone = timm.create_model(
            HP["backbone"],
            pretrained=pretrained,
            num_classes=0,
        )

        width = self.backbone.num_features
        dim = HP["concept_dim"]

        # Learn crop and disease features separately
        self.crop_encoder = nn.Sequential(
            nn.Linear(width, dim),
            nn.ReLU(),
        )
        self.disease_encoder = nn.Sequential(
            nn.Linear(width, dim),
            nn.ReLU(),
        )

        # Predict the main crop and disease tasks
        self.crop_head = nn.Linear(dim, NUM_CROPS)
        self.disease_head = nn.Linear(dim, NUM_DISEASES)

        # Predict the opposite tasks for adversarial training
        self.disease_from_crop = nn.Linear(dim, NUM_DISEASES)
        self.crop_from_disease = nn.Linear(dim, NUM_CROPS)

    def forward(self, images, use_adversary=False):
        shared = self.backbone(images)
        crop_features = self.crop_encoder(shared)
        disease_features = self.disease_encoder(shared)

        logits = (
            self.crop_head(crop_features),
            self.disease_head(disease_features),
        )

        adversary_logits = None
        if use_adversary:
            adversary_logits = (
                self.disease_from_crop(
                    GradientReversal.apply(crop_features)
                ),
                self.crop_from_disease(
                    GradientReversal.apply(disease_features)
                ),
            )

        return logits, adversary_logits, (crop_features, disease_features)


def orthogonal_loss(crop_features, disease_features):
    # Centre and normalise every feature dimension across the batch
    crop_features = F.normalize(
        crop_features - crop_features.mean(0),
        dim=0,
    )
    disease_features = F.normalize(
        disease_features - disease_features.mean(0),
        dim=0,
    )

    # Penalise cross correlation between crop and disease features
    return (crop_features.T @ disease_features).square().mean() 

# 4. Prediction and Feature Analysis

In [ ]:
def make_predict(model):
    @torch.no_grad()
    def predict(images):
        model.eval()
        (crop_logits, disease_logits), _, _ = model(images.to(DEVICE))

        crop_log_prob = F.log_softmax(crop_logits.float(), dim=1)
        disease_log_prob = F.log_softmax(disease_logits.float(), dim=1)

        # Combine crop and disease scores for the 333 valid pairs
        return (
            crop_log_prob[:, pairs[:, 0]]
            + disease_log_prob[:, pairs[:, 1]]
        )

    return predict


@torch.no_grad()
def measure_correlation(model, loader):
    model.eval()
    crop_batches = []
    disease_batches = []

    for images, _, _, _ in loader:
        _, _, (crop_features, disease_features) = model(images.to(DEVICE))
        crop_batches.append(crop_features.float().cpu())
        disease_batches.append(disease_features.float().cpu())

    crop_features = torch.cat(crop_batches)
    disease_features = torch.cat(disease_batches)
    return orthogonal_loss(crop_features, disease_features).item()

# 5. Training All Four Experiments

In [ ]:
CHECKPOINT_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

# Build and cache one identical starting state for all experiments
set_seed(42)
template_model = DisentangleNet(pretrained=True)
initial_state = {
    key: value.detach().cpu().clone()
    for key, value in template_model.state_dict().items()
}
del template_model

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

experiment_records = []
history_records = []

for name, orthogonal_weight, adversarial_weight in EXPERIMENTS:
    print(f"\nStarting {name}")
    set_seed(42)

    # Fresh loaders keep the experiment comparison reproducible
    train_loader, val_loader, _, _ = make_loaders(DEBUG)

    # Fresh model prevents information leaking between experiments
    model = DisentangleNet(pretrained=False)
    model.load_state_dict(initial_state)
    model = model.to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=HP["lr"],
        weight_decay=HP["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS,
    )
    criterion = nn.CrossEntropyLoss(
        label_smoothing=HP["label_smoothing"]
    )

    experiment_hp = {
        **HP,
        "orthogonal_weight": orthogonal_weight,
        "adversarial_weight": adversarial_weight,
        "mixed_precision": USE_AMP,
    }

    checkpoint_path = (
        CHECKPOINT_DIR
        / f"approach2_{name}_{RUN_TYPE}.pt"
    )
    best_val = -1.0
    best_epoch = 0
    predict = make_predict(model)

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0.0
        total_ce = 0.0
        total_adversarial = 0.0
        total_orthogonal = 0.0

        for images, crop_targets, disease_targets, _ in train_loader:
            images = images.to(DEVICE, non_blocking=True)
            crop_targets = crop_targets.to(DEVICE, non_blocking=True)
            disease_targets = disease_targets.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            # Use bfloat16 autocast on NVIDIA GPUs
            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.bfloat16,
                enabled=USE_AMP,
            ):
                (
                    (crop_logits, disease_logits),
                    adversary_logits,
                    (crop_features, disease_features),
                ) = model(
                    images,
                    use_adversary=adversarial_weight > 0,
                )

                classification_loss = (
                    criterion(crop_logits, crop_targets)
                    + criterion(disease_logits, disease_targets)
                )

                if adversarial_weight > 0:
                    adversarial_loss = (
                        criterion(
                            adversary_logits[0],
                            disease_targets,
                        )
                        + criterion(
                            adversary_logits[1],
                            crop_targets,
                        )
                    )
                else:
                    adversarial_loss = torch.zeros(
                        (),
                        device=DEVICE,
                    )

            if orthogonal_weight > 0:
                feature_loss = orthogonal_loss(
                    crop_features.float(),
                    disease_features.float(),
                )
            else:
                feature_loss = torch.zeros(
                    (),
                    device=DEVICE,
                )

            loss = (
                classification_loss
                + adversarial_weight * adversarial_loss
                + orthogonal_weight * feature_loss
            )

            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            total_ce += classification_loss.item()
            total_adversarial += adversarial_loss.item()
            total_orthogonal += feature_loss.item()

        scheduler.step()
        validation_top1 = val_accuracy(predict, val_loader)
        batch_count = len(train_loader)

        history_records.append({
            "run_type": RUN_TYPE,
            "experiment": name,
            "epoch": epoch + 1,
            "loss": total_loss / batch_count,
            "classification_loss": total_ce / batch_count,
            "adversarial_loss": total_adversarial / batch_count,
            "orthogonal_loss": total_orthogonal / batch_count,
            "val_top1": validation_top1,
        })

        if validation_top1 > best_val:
            best_val = validation_top1
            best_epoch = epoch + 1

            torch.save({
                "model_name": name,
                "state_dict": {
                    key: value.detach().cpu()
                    for key, value in model.state_dict().items()
                },
                "hp": experiment_hp,
                "best_val": best_val,
                "best_epoch": best_epoch,
                "run_type": RUN_TYPE,
            }, checkpoint_path)

        print(
            f"{name} | Epoch {epoch + 1}/{EPOCHS} | "
            f"Loss {total_loss / batch_count:.3f} | "
            f"Val Top-1 {validation_top1:.2f}%"
        )

    # Restore the best validation epoch
    saved = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True,
    )
    model.load_state_dict(saved["state_dict"])
    validation_correlation = measure_correlation(model, val_loader)

    experiment_records.append({
        "run_type": RUN_TYPE,
        "experiment": name,
        "orthogonal_weight": orthogonal_weight,
        "adversarial_weight": adversarial_weight,
        "best_epoch": best_epoch,
        "val_top1": best_val,
        "val_correlation": validation_correlation,
        "checkpoint": str(checkpoint_path),
    })

    del predict
    del model
    del optimizer
    del scheduler
    del train_loader
    del val_loader
    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

history = pd.DataFrame(history_records)
history.to_csv(
    RESULTS_DIR / f"approach2_history_{RUN_TYPE}.csv",
    index=False,
)

# 6. Compare Validation Results

In [ ]:
summary = pd.DataFrame(experiment_records)
summary = summary.sort_values(
    ["val_top1", "val_correlation"],
    ascending=[False, True],
).reset_index(drop=True)

summary.to_csv(
    RESULTS_DIR / f"approach2_validation_{RUN_TYPE}.csv",
    index=False,
)

display(summary)
validation_winner = summary.iloc[0]["experiment"]
print("Best validation experiment:", validation_winner)

# 7. Final Evaluation and Save

In [ ]:
if DEBUG:
    print("Debug complete")
    print("Set DEBUG to False then restart and run all for full training")

else:
    _, _, seen_loader, unseen_loader = make_loaders(False)
    official_records = []

    for name, _, _ in EXPERIMENTS:
        checkpoint_path = (
            CHECKPOINT_DIR
            / f"approach2_{name}_full.pt"
        )
        saved = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=True,
        )

        model = DisentangleNet(pretrained=False).to(DEVICE)
        model.load_state_dict(saved["state_dict"])
        predict = make_predict(model)

        results = evaluate(
            predict,
            seen_loader,
            unseen_loader,
            f"approach2_{name}",
            saved["hp"],
            robust=True,
        )

        official_records.append({
            "experiment": name,
            "best_epoch": saved["best_epoch"],
            "val_top1": saved["best_val"],
            "seen_top1": results["seen_top1"],
            "seen_top5": results["seen_top5"],
            "unseen_top1": results["unseen_top1"],
            "unseen_top5": results["unseen_top5"],
            "harmonic_mean": results["harmonic_mean"],
            "crop_acc": results["crop_acc"],
            "disease_acc": results["disease_acc"],
        })

        del predict
        del model
        gc.collect()

        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    official_summary = pd.DataFrame(official_records)
    official_summary = official_summary.sort_values(
        ["harmonic_mean", "unseen_top1", "seen_top1"],
        ascending=[False, False, False],
    ).reset_index(drop=True)

    official_summary.to_csv(
        RESULTS_DIR / "approach2_official_comparison.csv",
        index=False,
    )
    display(official_summary)

    final_winner = official_summary.iloc[0]["experiment"]
    final_checkpoint = torch.load(
        CHECKPOINT_DIR / f"approach2_{final_winner}_full.pt",
        map_location="cpu",
        weights_only=True,
    )

    # Save the selected model for team integration
    torch.save({
        "model_name": "approach2_disentangle",
        "selected_experiment": final_winner,
        "state_dict": final_checkpoint["state_dict"],
        "hp": final_checkpoint["hp"],
        "selection_metric": "harmonic_mean",
    }, CHECKPOINT_DIR / "approach2_disentangle.pt")

    print("Selected final experiment:", final_winner)